# 11-bob. 11-BOB. SI ASOSIDA AUTENTIFIKATSIYA VA FIRIBGARLIKNI ANIQLASH
## Amaliy misol: klaviatura dinamikasi bo'yicha foydalanuvchini tanish

[![Colab'da ochish](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mrmsi/si-kiberxavfsizlik/blob/main/11-bob/11-1_klaviatura-dinamikasi-boyicha-foydalanuvchini-tanish.ipynb)

*"Sun'iy intellektni kiberxavfsizlikda qo'llash" darsligi, 11-bob.*

Quyidagi dasturda Killurxi va Maksion tajribasiga o'xshash sun'iy ma'lumotlar yaratiladi. 20 nafar foydalanuvchi bir xil 10 belgili parolni yozadi. Har bir yozish 21 ta vaqt bilan tavsiflanadi: 11 ta tugmani (10 belgi va Enter) bosib turish vaqti va 10 ta tugmalar orasidagi vaqt. Barcha foydalanuvchilar parolni umumiy ritmda yozadi, lekin har birining bu ritmdan o'ziga xos farqi bor. Har bir foydalanuvchi uchun alohida detektor quriladi: u egasining yozishlaridagi har bir vaqtning odatiy qiymatini (mediana) va odatiy tebranishini o'rganadi. Yangi yozishning odatiy qiymatlardan o'lchamlangan Manxetten masofasi qancha katta bo'lsa, u shuncha shubhali hisoblanadi. Killurxi va Maksion tadqiqotida aynan shu usul eng yaxshi natija ko'rsatgan. Begonalar sifatida parolni biladigan qolgan 19 foydalanuvchining urinishlari olinadi. Ro'yxatga olishda foydalanuvchidan necha marta parol yozish so'ralishi natijaga qanday ta'sir qilishi ham tekshiriladi. Dastur amaliy mashg'ulotning A11 mavzusiga mos keladi.

**Ishga tushirish:** yuqoridagi menyudan *Runtime → Run all* (*Ishga tushirish → Hammasini bajarish*) ni tanlang yoki kataklarni birma-bir Shift+Enter bilan bajaring. Barcha ma'lumotlar dasturning o'zida yaratiladi, qo'shimcha fayl yuklash shart emas.

In [1]:
import numpy as np

rng = np.random.default_rng(2)
BELGILAR = 21   # 11 ta tugmani bosib turish vaqti + 10 ta tugmalar orasidagi vaqt (parol: 10 belgi + Enter)

# Bir xil parolni yozishning umumiy ritmi (soniya) va har bir foydalanuvchining undan o'ziga xos farqi
umumiy = np.concatenate([rng.uniform(0.07, 0.14, 11), rng.uniform(0.10, 0.40, 10)])
uslublar = {f"foydalanuvchi_{i:02d}": umumiy * rng.lognormal(0, 0.20, BELGILAR) for i in range(1, 21)}

def yozishlar(uslub, n):
    """Parolni n marta yozish: har safar biroz farqli tezlikda (sun'iy ma'lumot)."""
    kayfiyat = rng.normal(1.0, 0.10, (n, 1))                 # umumiy tezlik kundan-kunga o'zgaradi
    return uslub * kayfiyat * rng.lognormal(0, 0.17, (n, BELGILAR))

def teng_xato(haqiqiy, begona):
    """Teng xatolik darajasi (EER): rad etilgan egalar = qabul qilingan begonalar bo'lgan nuqta."""
    eng = (1.0, None)
    for ch in np.sort(np.concatenate([haqiqiy, begona])):
        frr, far = np.mean(haqiqiy > ch), np.mean(begona <= ch)
        eng = min(eng, (abs(frr - far), (frr + far) / 2))
    return eng[1]

def baholash(n_oqitish):
    """Har bir foydalanuvchi uchun detektor quriladi va EER hisoblanadi."""
    eerlar = []
    for nom, uslub in uslublar.items():
        royxat = yozishlar(uslub, n_oqitish + 100)
        oqitish, sinov = royxat[:n_oqitish], royxat[n_oqitish:]
        markaz = np.median(oqitish, axis=0)
        tarqoq = np.mean(np.abs(oqitish - markaz), axis=0)          # har bir belgining odatiy tebranishi
        masofa = lambda X: np.sum(np.abs(X - markaz) / tarqoq, axis=1)   # o'lchamlangan Manxetten masofasi
        # Begonalar: parolni biladigan boshqa 19 foydalanuvchi, har biri 5 urinish
        begonalar = np.vstack([yozishlar(u, 5) for n2, u in uslublar.items() if n2 != nom])
        eerlar.append(teng_xato(masofa(sinov), masofa(begonalar)))
    return np.array(eerlar)

for n in [5, 20, 100]:
    e = baholash(n)
    print(f"O'qitish uchun {n:>3} ta yozish: o'rtacha EER {e.mean():5.1%} "
          f"(eng yaxshi {e.min():.1%}, eng yomon {e.max():.1%})")

O'qitish uchun   5 ta yozish: o'rtacha EER 11.3% (eng yaxshi 5.1%, eng yomon 23.1%)
O'qitish uchun  20 ta yozish: o'rtacha EER  9.2% (eng yaxshi 3.1%, eng yomon 20.0%)
O'qitish uchun 100 ta yozish: o'rtacha EER  8.3% (eng yaxshi 4.1%, eng yomon 11.8%)


## Kitobdagi natija

Dasturni ishga tushirganingizda quyidagiga o'xshash natija chiqishi kerak (tasodifiy sonlar generatori urug'i qat'iy belgilangani uchun natija odatda aynan bir xil bo'ladi; faqat bajarilish vaqti va har safar yangi yaratiladigan tasodifiy kalitlar farq qilishi mumkin):

```
O'qitish uchun   5 ta yozish: o'rtacha EER 11.3% (eng yaxshi 5.1%, eng yomon 23.1%)
O'qitish uchun  20 ta yozish: o'rtacha EER  9.2% (eng yaxshi 3.1%, eng yomon 20.0%)
O'qitish uchun 100 ta yozish: o'rtacha EER  8.3% (eng yaxshi 4.1%, eng yomon 11.8%)
```

## O'zingiz sinab ko'ring

Parametrlarni (ma'lumotlar hajmi, chegara qiymati, tasodifiy son urug'i) o'zgartirib, natija qanday o'zgarishini kuzating. Kitobdagi 11-bobning B darajadagi topshiriqlari aynan shu daftar asosida bajariladi.